# **STEP 1: Environment Setup & Ecosystem Basics**

# Research/Learn:
Hugging Face Hub: A central repository for hosting models and datasets, functioning similarly to GitHub for Machine Learning.

Core Libraries:

*  transformers: The primary library for downloading and using pre-trained models.

*  tokenizers: A library to convert raw data into numerical tensors that machines can process.

*  datasets: A library for accessing and managing training data for various AI models.

*  accelerate: A tool to simplify and speed up training across different hardware configurations.

*  peft: Implements LoRA (Low-Rank Adaptation) for efficient and fast model fine-tuning.

*  trl: A library for Transformer Reinforcement Learning.

*  bitsandbytes: A library for model quantization (compressing models to 8-bit or 4-bit) to run smoothly on limited GPU memory.

Google Colab: A cloud-based platform for running Jupyter Notebooks with GPU support.

Authentication: Using an Access Token to securely authenticate with Hugging Face without manual password entry

Because I have a trouble prolbem with the Billing issue in Google Cloud Platform, I use collab to make notebook for step 1
#Actions:

In [ ]:
from huggingface_hub import login, upload_folder
login()
upload_folder(folder_path=".", repo_id="duyb2207513/step1", repo_type="model")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/vectorizer.joblib  : 100%|##########|  182kB /  182kB            

  /content/model.joblib       : 100%|##########|  161kB /  161kB            

  ...ata/mnist_train_small.csv: 100%|##########| 36.5MB / 36.5MB            

  ...ample_data/mnist_test.csv: 100%|##########| 18.3MB / 18.3MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/duyb2207513/step1/commit/cdc7bd40797e5c858ad4a17a2dd39e830a6ca920', commit_message='Upload folder using huggingface_hub', commit_description='', oid='cdc7bd40797e5c858ad4a17a2dd39e830a6ca920', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/step1', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/step1'), pr_revision=None, pr_num=None)

Cài đặt các thư viện chính:
Because colab have all of those library, so i don't install again.

In [ ]:

# !pip install transformers datasets tokenizers accelerate peft trl bitsandbytes huggingface_hub


 Successful login to HF and running basic inference from a Hub model
 # Milestone 1:

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Define the model name (A model already fine-tuned for sentiment analysis)
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 2. Provide a sample sentence (e.g., a compliment)
text = "This movie is absolutely amazing! I loved it."

# Tokenize the sentence and return PyTorch tensors
inputs = tokenizer(text, return_tensors="pt")

# 3. Feed the input into the model
with torch.no_grad():
    outputs = model(**inputs)

# 4. DECODE THE OUTPUT
# Extract the raw mathematical scores calculated by the model
logits = outputs.logits
print("1. Raw scores (Logits):", logits.tolist())

# Convert raw scores into probabilities (percentages) using the Softmax function
probabilities = torch.nn.functional.softmax(logits, dim=-1)
print("2. Probabilities (%):", probabilities.tolist())
# Expected output format: [[0.0001, 0.9999]] (meaning 0.01% Negative, 99.99% Positive)

# Select the index (ID) with the highest probability score using argmax
predicted_class_id = torch.argmax(probabilities).item()

# Map the predicted ID back to a human-readable string label using the model's configuration
predicted_label = model.config.id2label[predicted_class_id]

# Print the final result
print(f"\nInput sentence: '{text}'")
print(f"AI conclusion: {predicted_label}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1. Raw scores (Logits): [[-4.351162910461426, 4.709738731384277]]
2. Probabilities (%): [[0.0001161047475761734, 0.9998838901519775]]

Input sentence: 'This movie is absolutely amazing! I loved it.'
AI conclusion: POSITIVE


 # Milestone 2:

In [ ]:
# Pipeline:

#Run inference basic:
from transformers import pipeline
import torch

# 1 model example
model_id = "google/gemma-2-2b-it"

# 2. Make an pipeline for conversation
# pineline will automatic download model
pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda", # Chạy bằng GPU cho nhanh
)

# 3. Give a question like this:
messages = [
    {"role": "user", "content": "Chào bạn, hãy giới thiệu ngắn gọn về Hugging Face bằng tiếng Việt."},
]
outputs = pipe(messages, max_new_tokens=256)

# 4. the result
print(outputs[0]["generated_text"][-1]["content"])

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Hugging Face: Nền tảng cho trí tuệ nhân tạo

Hugging Face là một nền tảng trực tuyến cho việc phát triển và chia sẻ các mô hình trí tuệ nhân tạo (AI). Nó cung cấp một kho repository phong phú với:

* **Mô hình AI:** Từ các mô hình ngôn ngữ, phân loại hình ảnh, đến các thuật toán xử lý âm thanh, bạn có thể tìm kiếm và sử dụng các mô hình AI đã được tạo ra.
* **Tạo dựng mô hình:**  Sử dụng công cụ mạnh mẽ trong Hugging Face để xây dựng và fine-tuning các mô hình AI riêng của bạn.
* **Chia sẻ và kết nối:** Chia sẻ mã nguồn và mô hình với cộng đồng, trao đổi kinh nghiệm và thu thập ý tưởng.


Hugging Face giúp cho việc sử dụng AI trở nên dễ dàng hơn, mở ra cánh cửa cho các nhà phát triển và người dùng, từ dân số bình dân đến các doanh nghiệp lớn. 



# **STEP 2 Standard Training Pipelines for ML Models**

# Research/Learn:
A standard Machine Learning (ML) training workflow using Hugging Face (HF) involves loading datasets with datasets, pre-processing using tokenizers, fine-tuning models via transformers.Trainer, and evaluating performance. This iterative process optimizes models using pre-trained weights for specific tasks, utilizing GPU acceleration for efficiency. Steps in the HF Workflow:
* Data Preparation: Load datasets using the HF Datasets library and preprocess data (e.g., tokenization) using HF Tokenizers to convert text into numerical input suitable for models.
* Model Selection: Select a pre-trained model architecture from HF Transformers appropriate for the task (e.g., classification, generation).
* Training Setup: Utilize the Trainer API to define training arguments such as learning rate, batch size, and evaluation strategies.
* Training: Fine-tune the model on your dataset.Evaluation: Assess the model's performance on a validation set, commonly tracking metrics like accuracy, F1-score, or perplexity.
* Sharing: Push the final model to the Hugging Face Hub to share or deploy

Differences from DL and LLM is:Deep Learning is child of Machine Learning, using a lot of neural network layer to simulate how brain process data. LLM (Large Language Model) is application of Deep learning that using a special neural network architecture is called Transformer to process text.

Data preprocessing: is the progress that make cleaned data.

Evaluation metrics are used to measure how well a machine learning model performs on a given task. Different tasks require different metrics. For example, with clasification tasks use accuracy, F1-Score,; with regression tasks use MSE,MAE,.; with seq2seq tasks use rouge or perplexity,...

Trainer API in Hugging Face simplifies the training and evaluation process of transformer models by providing a high-level interface.

In [ ]:
# Example about Trainer API
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
#Action1: Load dataset
from datasets import load_dataset
# load iris classification dataset
dataset = load_dataset("hitorilabs/iris")

In [ ]:
dataset['train']


Dataset({
    features: ['petal_length', 'petal_width', 'sepal_length', 'sepal_width', 'species'],
    num_rows: 150
})

In [ ]:
train_ds = dataset['train']

In [ ]:
#Action 2:normalize to [0,1]
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

def normalize_dataset(dataset):
    # Convert HF Dataset -> pandas
    df = dataset.to_pandas()

    feature_cols = ['petal_length', 'petal_width', 'sepal_length', 'sepal_width']

    # Khởi tạo scaler
    scaler = MinMaxScaler()

    # Fit + transform
    df[feature_cols] = scaler.fit_transform(df[feature_cols])

    return df, scaler
# Convert sang pandas
df = train_ds.to_pandas()

# Feature columns
feature_cols = ['petal_length', 'petal_width', 'sepal_length', 'sepal_width']

# Normalize
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print(df.head())

   petal_length  petal_width  sepal_length  sepal_width  species
0      0.067797     0.041667      0.222222     0.625000        0
1      0.067797     0.041667      0.166667     0.416667        0
2      0.050847     0.041667      0.111111     0.500000        0
3      0.084746     0.041667      0.083333     0.458333        0
4      0.067797     0.041667      0.194444     0.666667        0


In [ ]:

#Action3 : select model, I load native Bayes model
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
#split dataset:
X_train, X_test, y_train, y_test = train_test_split(df[feature_cols], df['species'], test_size=0.2, random_state=42)
#Action 4: training model
model=GaussianNB()
model.fit(X_train, y_train)




GaussianNB()

In [ ]:
#4 evaluating model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 1.0


# Milestone:

Fully trained ML model (e.g., text classifier) pushed to HF Hub with
model card.

In [ ]:

from datasets import load_dataset
dataset= load_dataset("ag_news")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
texts = dataset["train"]["text"]
labels = dataset["train"]["label"]

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(texts)
y = labels

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X, y)

LogisticRegression(max_iter=200)

In [ ]:
from sklearn.metrics import accuracy_score

X_test = vectorizer.transform(dataset["test"]["text"])
y_test = dataset["test"]["label"]

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.9063157894736842


In [ ]:
from huggingface_hub import HfApi
import joblib

# Save model
joblib.dump(model, "model.joblib")
joblib.dump(vectorizer, "vectorizer.joblib")

api = HfApi()


repo_type="model"
api.upload_file(
    path_or_fileobj="model.joblib",
    path_in_repo="model.joblib",
    repo_id="duyb2207513/step1"
)

api.upload_file(
    path_or_fileobj="vectorizer.joblib",
    path_in_repo="vectorizer.joblib",
    repo_id="duyb2207513/step1"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  model.joblib                : 100%|##########|  161kB /  161kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  vectorizer.joblib           : 100%|##########|  182kB /  182kB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/duyb2207513/step1/commit/cdc7bd40797e5c858ad4a17a2dd39e830a6ca920', commit_message='Upload vectorizer.joblib with huggingface_hub', commit_description='', oid='cdc7bd40797e5c858ad4a17a2dd39e830a6ca920', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/step1', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/step1'), pr_revision=None, pr_num=None)